# DTM Ground Classification — Kaggle GPU
### Target: >95% Accuracy | Budget: 12-hour Kaggle session

**Run order: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8**  
All names are consistent across every cell in this notebook.

| Cell | What it does | Key names defined |
|---|---|---|
| 1 | GPU check + seed | `DEVICE`, `SESSION_START` |
| 2 | Config | `CFG` |
| 3 | Feature cache build | `geo_features()`, `build_feat_cache()` |
| 4 | Model definition | `DTMPointNet2`, `model` |
| 5 | Dataset class | `GroundDataset` |
| 6 | Training loop | `train()`, `history`, `best_acc` |
| 7 | Threshold sweep | `optimal_threshold.json` |
| 8 | Package outputs | `dtm_outputs.zip` |


In [ ]:
# Cell 1 — GPU check + reproducibility seed
import torch, random, time, os
import numpy as np

# ── GPU ──────────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected.\n'
        'Kaggle -> Settings -> Accelerator -> GPU T4 x2 -> Save\n'
        'Run -> Restart Session -> Run All Cells'
    )

props = torch.cuda.get_device_properties(0)
sm    = props.major * 10 + props.minor
name  = torch.cuda.get_device_name(0)

# Probe: surfaces stale-context errors immediately
_p = torch.zeros(4, device='cuda') + 1
torch.cuda.synchronize()
del _p

if sm < 70:
    raise RuntimeError(
        f'GPU {name} (sm_{sm}) is not supported.\n'
        'Need T4 (sm_75) or better.\n'
        'Kaggle -> Settings -> Accelerator -> GPU T4 -> Save'
    )

DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

print(f'GPU  : {name}  (sm_{sm})')
print(f'VRAM : {props.total_memory/1e9:.1f} GB')

# ── Seed ─────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Session clock (used by Cell 6 to respect 12-hour Kaggle limit) ────────────
SESSION_START    = time.time()
SESSION_BUDGET_H = 11.0   # stop 1 h before hard 12-hour limit

print(f'Seed          : {SEED}')
print(f'Session clock : started')
print('Cell 1 done -- proceed to Cell 2')


In [ ]:
# Cell 2 — Configuration
from pathlib import Path

# ── Auto-detect dataset location ─────────────────────────────────────────────
DATASET_ROOT = '/kaggle/input/point-cloud-data-of-10-indian-villages'

def _find_split_root(base):
    base = Path(base)
    # Direct train/val folders
    if (base / 'train').exists() and (base / 'val').exists():
        return str(base), None
    # One level deeper
    for sub in sorted(base.iterdir()):
        if sub.is_dir() and (sub / 'train').exists():
            return str(sub), None
    # Zip file
    zips = list(base.rglob('training_data.zip'))
    if zips:
        return None, str(zips[0])
    return str(base), None

_TDIR, _ZPATH = _find_split_root(DATASET_ROOT)
_USE_ZIP = _ZPATH is not None and _TDIR is None

CFG = {
    # ── Data ──────────────────────────────────────────────────────────────────
    'use_zip'          : _USE_ZIP,
    'zip_path'         : _ZPATH or '',
    'training_dir'     : _TDIR  or '',
    'feat_cache_dir'   : '/kaggle/working/feat_cache',

    # ── Outputs ───────────────────────────────────────────────────────────────
    'logs_dir'         : '/kaggle/working/logs',
    'model_save_path'  : '/kaggle/working/logs/best_model.pth',
    'swa_save_path'    : '/kaggle/working/logs/swa_model.pth',
    'ckpt_path'        : '/kaggle/working/logs/checkpoint.pth',
    'history_path'     : '/kaggle/working/logs/history.json',
    'curves_path'      : '/kaggle/working/logs/training_curves.png',
    'threshold_path'   : '/kaggle/working/logs/optimal_threshold.json',

    # ── Points ────────────────────────────────────────────────────────────────
    'max_pts'          : 4096,
    'seed'             : 42,

    # ── Training ──────────────────────────────────────────────────────────────
    'epochs'           : 100,
    'batch_size'       : 8,
    'grad_accum'       : 4,        # effective batch = 32
    'max_train_tiles'  : 99999,
    'max_val_tiles'    : 99999,

    # ── Optimiser ─────────────────────────────────────────────────────────────
    'max_lr'           : 0.01,
    'weight_decay'     : 1e-4,
    'grad_clip'        : 5.0,

    # ── Focal Loss ────────────────────────────────────────────────────────────
    'focal_alpha'      : 0.75,     # weight for minority ground class
    'focal_gamma'      : 2.0,      # focusing exponent
    'label_smooth'     : 0.05,     # label smoothing epsilon

    # ── SWA ───────────────────────────────────────────────────────────────────
    'swa_start'        : 70,       # epoch to begin stochastic weight averaging
    'swa_lr'           : 0.001,

    # ── Schedule ──────────────────────────────────────────────────────────────
    'val_every'        : 2,        # run validation every N epochs
    'patience'         : 18,       # early-stop patience (in val checks)
    'use_amp'          : True,     # automatic mixed precision
    'resume'           : True,     # auto-resume from checkpoint if it exists

    # ── DataLoader ────────────────────────────────────────────────────────────
    'num_workers'      : 4,
    'prefetch_factor'  : 2,
}

for _d in [CFG['logs_dir'], CFG['feat_cache_dir']]:
    Path(_d).mkdir(parents=True, exist_ok=True)

print(f'Data source    : {"ZIP: " + str(_ZPATH) if _USE_ZIP else _TDIR}')
print(f'Epochs         : {CFG["epochs"]}')
print(f'Effective batch: {CFG["batch_size"]} x {CFG["grad_accum"]} = '
      f'{CFG["batch_size"]*CFG["grad_accum"]}')
print(f'SWA from epoch : {CFG["swa_start"]}')
print('Cell 2 done -- proceed to Cell 3')


In [ ]:
# Cell 3 — Geospatial Feature Engineering + Cache Build
#
# Computes 4 terrain features per point:
#   dZ        = height above local 2m-grid minimum  (0 for ground, >0 for objects)
#   roughness = Z std-dev in 2m cell                (low for bare earth)
#   slope     = max |dZ| across 4 neighbours/2m     (steep = wall / tree edge)
#   density   = normalised point count per cell     (high = canopy)
# All 4 are z-score normalised before saving.
# Cache is built ONCE (~10 min). Every subsequent epoch loads in <1 ms.
#
# KEY: function is named  geo_features()  throughout this notebook.

import numpy as np
from pathlib import Path, PurePosixPath
from tqdm import tqdm
import time, io, zipfile


def geo_features(xyz: np.ndarray) -> np.ndarray:
    """
    Input : (N, 3) float32 XYZ
    Output: (N, 4) float32 [dZ, roughness, slope, density]  z-score normalised
    """
    xyz = xyz.astype(np.float32, copy=False)
    x_min = float(xyz[:, 0].min())
    y_min = float(xyz[:, 1].min())
    x_rng = float(xyz[:, 0].max() - x_min) + 1e-6
    y_rng = float(xyz[:, 1].max() - y_min) + 1e-6

    GW = int(np.clip(x_rng / 2.0, 16, 64))
    GH = int(np.clip(y_rng / 2.0, 16, 64))
    NC = GW * GH

    xi = np.clip(((xyz[:, 0] - x_min) / x_rng * GW).astype(np.int32), 0, GW-1)
    yi = np.clip(((xyz[:, 1] - y_min) / y_rng * GH).astype(np.int32), 0, GH-1)
    ci = yi * GW + xi
    z  = xyz[:, 2]

    c_min = np.full(NC, np.inf,  dtype=np.float32)
    c_sum = np.zeros(NC,         dtype=np.float32)
    c_sq  = np.zeros(NC,         dtype=np.float32)
    c_cnt = np.zeros(NC,         dtype=np.float32)
    np.minimum.at(c_min, ci, z)
    np.add.at(c_sum, ci, z)
    np.add.at(c_sq,  ci, z * z)
    np.add.at(c_cnt, ci, 1.0)

    empty          = c_cnt == 0
    c_cnt[empty]   = 1.0
    c_min[empty]   = z.mean()
    c_sum[empty]   = z.mean()
    c_sq[empty]    = z.mean() ** 2

    c_mean = c_sum / c_cnt
    c_std  = np.sqrt(np.maximum(c_sq / c_cnt - c_mean ** 2, 0.0))

    # dZ: height above local minimum
    dz        = np.clip(z - c_min[ci], 0.0, None)
    # roughness: local Z standard deviation
    roughness = c_std[ci]
    # slope: max gradient across 4 neighbours
    cg  = c_min.reshape(GH, GW)
    pad = np.pad(cg, 1, mode='edge')
    sg  = np.max(np.stack([
        np.abs(cg - pad[:-2, 1:-1]),
        np.abs(cg - pad[2:,  1:-1]),
        np.abs(cg - pad[1:-1, :-2]),
        np.abs(cg - pad[1:-1, 2:]),
    ]), axis=0) / 2.0
    slope   = sg.reshape(-1)[ci]
    # density: normalised point count
    density = c_cnt[ci] / (c_cnt.max() + 1e-6)

    feat = np.stack([dz, roughness, slope, density], axis=1)
    mu   = feat.mean(axis=0, keepdims=True)
    sig  = feat.std(axis=0,  keepdims=True) + 1e-6
    return ((feat - mu) / sig).astype(np.float32)


def build_feat_cache(cfg: dict, split: str):
    """
    Pre-compute geo_features() for every tile in `split` and save to disk.
    Skips tiles that already have a cache file (safe to re-run).
    """
    cache_dir = Path(cfg['feat_cache_dir']) / split
    cache_dir.mkdir(parents=True, exist_ok=True)

    if cfg['use_zip']:
        with zipfile.ZipFile(cfg['zip_path']) as zf:
            names = zf.namelist()
        tiles = {}
        for n in names:
            parts = PurePosixPath(n).parts
            if split in parts and n.endswith('points.npy'):
                tile = parts[parts.index(split) + 1]
                tiles[tile] = n
        zf_h = zipfile.ZipFile(cfg['zip_path'])
        try:
            for tile, zp in tqdm(tiles.items(),
                                  desc=f'Cache [{split}]', unit='tile'):
                out = cache_dir / f'{tile}.npy'
                if out.exists():
                    continue
                with zf_h.open(zp) as fh:
                    xyz = np.load(io.BytesIO(fh.read()))
                np.save(str(out), geo_features(xyz))
        finally:
            zf_h.close()
    else:
        split_dir = Path(cfg['training_dir']) / split
        tiles = sorted(split_dir.glob('tile_*'))
        for td in tqdm(tiles, desc=f'Cache [{split}]', unit='tile'):
            out = cache_dir / f'{td.name}.npy'
            if out.exists():
                continue
            xyz = np.load(str(td / 'points.npy'))
            np.save(str(out), geo_features(xyz))


print('Building feature caches (one-time, ~10 min for 20k tiles)...')
t0 = time.time()
build_feat_cache(CFG, 'train')
build_feat_cache(CFG, 'val')
print(f'Cache ready in {(time.time()-t0)/60:.1f} min')
print('Cell 3 done -- proceed to Cell 4')


In [ ]:
# Cell 4 — PointNet++ MSG Model
# KEY: class is named  DTMPointNet2  throughout this notebook.

import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Geometric primitives ──────────────────────────────────────────────────────

def _fps(xyz: torch.Tensor, n: int) -> torch.Tensor:
    B, N, _ = xyz.shape
    dev  = xyz.device
    cent = torch.zeros(B, n, dtype=torch.long, device=dev)
    dist = torch.full((B, N), 1e10, dtype=torch.float32, device=dev)
    far  = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    bi   = torch.arange(B, dtype=torch.long, device=dev)
    for i in range(n):
        cent[:, i] = far
        c    = xyz[bi, far].unsqueeze(1)
        d    = ((xyz - c) ** 2).sum(-1)
        dist = torch.where(d < dist, d, dist)
        far  = dist.argmax(-1)
    return cent


def _idx(pts: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    B  = pts.shape[0]
    bi = torch.arange(B, device=pts.device)
    bi = bi.view([B] + [1] * (idx.dim() - 1)).expand_as(idx)
    return pts[bi, idx]


def _ball(new_xyz: torch.Tensor, xyz: torch.Tensor,
          r: float, k: int) -> torch.Tensor:
    d = torch.cdist(new_xyz, xyz)
    m = torch.where(d <= r, d, d.new_full((), 1e10))
    return m.topk(k, dim=-1, largest=False).indices


# ── Layers ────────────────────────────────────────────────────────────────────

class _MSG(nn.Module):
    """Multi-Scale Grouping Set Abstraction."""
    def __init__(self, n_ctr, radii, nsamps, in_ch, specs):
        super().__init__()
        self.n_ctr  = n_ctr
        self.radii  = radii
        self.nsamps = nsamps
        self.mlps   = nn.ModuleList()
        for dims in specs:
            layers, ch = [], in_ch + 3
            for d in dims:
                layers += [nn.Conv2d(ch, d, 1, bias=False),
                           nn.BatchNorm2d(d), nn.ReLU(True)]
                ch = d
            self.mlps.append(nn.Sequential(*layers))

    def forward(self, xyz, feat):
        ci   = _fps(xyz, self.n_ctr)
        nxyz = _idx(xyz, ci)
        outs = []
        for r, k, mlp in zip(self.radii, self.nsamps, self.mlps):
            idx  = _ball(nxyz, xyz, r, k)
            grp  = _idx(feat, idx)
            rxyz = _idx(xyz, idx) - nxyz.unsqueeze(2)
            grp  = torch.cat([rxyz, grp], -1).permute(0, 3, 2, 1)
            outs.append(mlp(grp).max(2).values)
        return nxyz, torch.cat(outs, 1).permute(0, 2, 1)


class _SA(nn.Module):
    """Single-scale Set Abstraction (global SA3 level)."""
    def __init__(self, r, k, in_ch, dims):
        super().__init__()
        self.r = r; self.k = k
        layers, ch = [], in_ch + 3
        for d in dims:
            layers += [nn.Conv2d(ch, d, 1, bias=False),
                       nn.BatchNorm2d(d), nn.ReLU(True)]
            ch = d
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, feat):
        n_ctr = max(1, xyz.shape[1] // 4)
        ci    = _fps(xyz, n_ctr)
        nxyz  = _idx(xyz, ci)
        idx   = _ball(nxyz, xyz, self.r, self.k)
        grp   = _idx(feat, idx)
        rxyz  = _idx(xyz, idx) - nxyz.unsqueeze(2)
        grp   = torch.cat([rxyz, grp], -1).permute(0, 3, 2, 1)
        return nxyz, self.mlp(grp).max(2).values.permute(0, 2, 1)


class _FP(nn.Module):
    """Feature Propagation — IDW interpolation from sparse to dense."""
    def __init__(self, in_ch, dims):
        super().__init__()
        layers, ch = [], in_ch
        for d in dims:
            layers += [nn.Conv1d(ch, d, 1, bias=False),
                       nn.BatchNorm1d(d), nn.ReLU(True)]
            ch = d
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz1, xyz2, f1, f2):
        d    = torch.cdist(xyz1, xyz2)
        t3   = d.topk(3, dim=-1, largest=False)
        w    = 1.0 / (t3.values + 1e-8)
        w    = w / w.sum(-1, keepdim=True)
        interp = (_idx(f2, t3.indices) * w.unsqueeze(-1)).sum(2)
        feat   = torch.cat([f1, interp], -1) if f1 is not None else interp
        return self.mlp(feat.permute(0, 2, 1)).permute(0, 2, 1)


# ── Full model ────────────────────────────────────────────────────────────────

class DTMPointNet2(nn.Module):
    """
    PointNet++ MSG — binary ground classification.
    Input : (B, N, 7)  [x, y, z, dZ, roughness, slope, density]
    Output: (B, N, 2)  logits [non-ground, ground]
    """
    def __init__(self, in_feat: int = 7):
        super().__init__()
        C = in_feat
        # SA1: small r=0.5m catches kuccha wall texture
        #      large r=1.5m sees wall + compound boundary together
        self.sa1 = _MSG(512, [0.5, 1.5], [32, 64],   C,
                        [[32, 64], [64, 128]])         # -> 192-ch
        self.sa2 = _MSG(128, [3.0, 6.0], [64, 128],  192,
                        [[128, 128], [128, 256]])       # -> 384-ch
        self.sa3 = _SA(12.0, 128, 384, [256, 512])    # -> 512-ch
        self.fp3 = _FP(512 + 384, [256, 256])
        self.fp2 = _FP(256 + 192, [256, 128])
        self.fp1 = _FP(128 + C,   [128, 128])
        self.head = nn.Sequential(
            nn.Conv1d(128, 128, 1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(True),
            nn.Dropout(0.4),
            nn.Conv1d(128, 2, 1),
        )

    def forward(self, pts: torch.Tensor) -> torch.Tensor:
        xyz0 = pts[:, :, :3].contiguous()
        f0   = pts.contiguous()
        xyz1, f1 = self.sa1(xyz0, f0)
        xyz2, f2 = self.sa2(xyz1, f1)
        xyz3, f3 = self.sa3(xyz2, f2)
        f2 = self.fp3(xyz2, xyz3, f2, f3)
        f1 = self.fp2(xyz1, xyz2, f1, f2)
        f0 = self.fp1(xyz0, xyz1, f0, f1)
        return self.head(f0.permute(0, 2, 1)).permute(0, 2, 1)


# ── Sanity check ──────────────────────────────────────────────────────────────
model = DTMPointNet2(in_feat=7).to(DEVICE)
_x    = torch.randn(2, 256, 7, device=DEVICE)
with torch.no_grad():
    _out = model(_x)
assert _out.shape == (2, 256, 2), f'Bad shape: {_out.shape}'
del _x, _out
torch.cuda.empty_cache()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model : DTMPointNet2   params={n_params/1e6:.2f} M')
print(f'Output shape check : (2, 256, 2)  PASSED')
print('Cell 4 done -- proceed to Cell 5')


In [ ]:
# Cell 5 — Dataset Class
# KEY: class is named  GroundDataset  throughout this notebook.
# Uses geo_features() defined in Cell 3.

import io, zipfile
import numpy as np
import torch
from pathlib import Path, PurePosixPath
from torch.utils.data import Dataset


class GroundDataset(Dataset):
    """
    Loads LiDAR tiles (points.npy + labels.npy).
    Returns 7-channel feature tensor: [x, y, z, dZ, roughness, slope, density]

    7 augmentations applied during training:
      1. Z-axis rotation     -- terrain is rotationally symmetric
      2. Gaussian Z jitter   -- LiDAR measurement noise
      3. Uniform XYZ scale   -- sensor altitude variation
      4. Random XY flip      -- data doubling at no cost
      5. Anisotropic XY scale-- elongated village layouts
      6. Z-only stretch      -- variable tree/building heights
      7. Point dropout 10%%  -- sparse scan patches
    """

    def __init__(self, cfg: dict, split: str = 'train',
                 augment: bool = False, max_tiles: int = None):
        self.augment  = augment
        self.max_pts  = cfg['max_pts']
        self.seed     = cfg['seed']
        self.split    = split
        self.use_zip  = cfg['use_zip']

        # Feature cache directory (pre-computed by Cell 3)
        self._cache = Path(cfg['feat_cache_dir']) / split

        if self.use_zip:
            self._zpath = cfg['zip_path']
            self._tiles = self._discover_zip()
        else:
            self._tdir  = Path(cfg['training_dir']) / split
            self._tiles = sorted(
                p.name for p in self._tdir.glob('tile_*')
                if (p / 'labels.npy').exists()
            )

        if max_tiles and len(self._tiles) > max_tiles:
            rng  = np.random.default_rng(self.seed)
            pick = rng.choice(len(self._tiles), size=max_tiles, replace=False)
            self._tiles = [self._tiles[i] for i in sorted(pick)]

        print(f'  [{split}] {len(self._tiles)} tiles  '
              f'| cache: {self._cache.exists()}')

    def _discover_zip(self):
        with zipfile.ZipFile(self._zpath) as zf:
            names = zf.namelist()
        tiles = set()
        for n in names:
            parts = PurePosixPath(n).parts
            if self.split in parts and 'tile_' in parts[-2]:
                tiles.add(parts[parts.index(self.split) + 1])
        return sorted(tiles)

    def _load_zip_tile(self, tile):
        if not hasattr(self, '_zf') or self._zf is None:
            self._zf = zipfile.ZipFile(self._zpath)
        def _rd(fn):
            with self._zf.open(f'{self.split}/{tile}/{fn}') as fh:
                return np.load(io.BytesIO(fh.read()))
        return _rd('points.npy'), _rd('labels.npy')

    def __len__(self):
        return len(self._tiles)

    def __getitem__(self, idx: int):
        tile = self._tiles[idx]
        rng  = np.random.default_rng(idx + self.seed * 10000)

        if self.use_zip:
            xyz, labels = self._load_zip_tile(tile)
        else:
            xyz    = np.load(str(self._tdir / tile / 'points.npy'))
            labels = np.load(str(self._tdir / tile / 'labels.npy'))

        # ── 7 augmentations (train only) ──────────────────────────────────────
        if self.augment:
            t = rng.uniform(0, 2 * np.pi)
            R = np.array([[np.cos(t), -np.sin(t)],
                          [np.sin(t),  np.cos(t)]], dtype=np.float32)
            xyz[:, :2] = xyz[:, :2] @ R.T                        # 1. rotate
            xyz[:, 2] += rng.normal(0, 0.02, len(xyz)).astype(np.float32)  # 2. jitter
            xyz        *= rng.uniform(0.95, 1.05)                # 3. uniform scale
            if rng.random() > 0.5: xyz[:, 0] *= -1              # 4a. X flip
            if rng.random() > 0.5: xyz[:, 1] *= -1              # 4b. Y flip
            xyz[:, 0]  *= rng.uniform(0.9, 1.1)                 # 5a. aniso X
            xyz[:, 1]  *= rng.uniform(0.9, 1.1)                 # 5b. aniso Y
            xyz[:, 2]  *= rng.uniform(0.9, 1.1)                 # 6. Z stretch
            keep = rng.random(len(xyz)) > 0.10                  # 7. dropout
            if keep.sum() > 64:
                xyz    = xyz[keep]
                labels = labels[keep]

        # ── Subsample / pad to fixed size ─────────────────────────────────────
        N = len(xyz)
        if N > self.max_pts:
            ch  = rng.choice(N, self.max_pts, replace=False)
            xyz = xyz[ch]; labels = labels[ch]
        elif N < self.max_pts:
            pad = rng.choice(N, self.max_pts - N, replace=True)
            xyz    = np.concatenate([xyz,    xyz[pad]])
            labels = np.concatenate([labels, labels[pad]])

        # ── Geospatial features ───────────────────────────────────────────────
        # Use cached features when available and not augmenting
        cache_path = self._cache / f'{tile}.npy'
        if cache_path.exists() and not self.augment:
            feat4 = np.load(str(cache_path))
            if len(feat4) != self.max_pts:
                feat4 = geo_features(xyz)    # size mismatch -> recompute
        else:
            feat4 = geo_features(xyz)        # augmented -> always recompute

        # ── Normalise XYZ to unit cube ────────────────────────────────────────
        xn = xyz.copy()
        xn[:, :2] -= xn[:, :2].mean(axis=0)
        xn[:, 2]  -= float(np.median(xn[:, 2]))
        xn /= (np.abs(xn).max() + 1e-6)

        pts7 = np.concatenate([xn, feat4], axis=1).astype(np.float32)
        return (torch.from_numpy(pts7),
                torch.from_numpy(labels.astype(np.int64)))


# ── Quick smoke test ──────────────────────────────────────────────────────────
_ds = GroundDataset(CFG, 'train', augment=False, max_tiles=2)
_pts, _lbl = _ds[0]
assert _pts.shape == (CFG['max_pts'], 7), f'Bad pts shape: {_pts.shape}'
assert _lbl.shape == (CFG['max_pts'],),   f'Bad lbl shape: {_lbl.shape}'
del _ds, _pts, _lbl

print(f'GroundDataset smoke test PASSED  (shape {CFG["max_pts"]} x 7)')
print('Cell 5 done -- proceed to Cell 6')


In [ ]:
# Cell 6 — Training Loop
# Uses: CFG, DEVICE, SESSION_START, SESSION_BUDGET_H  (Cell 1+2)
#       geo_features()   (Cell 3)
#       DTMPointNet2     (Cell 4)
#       GroundDataset    (Cell 5)

import contextlib, time, json, math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

# ── Dependency guard ──────────────────────────────────────────────────────────
_NEED = {
    'CFG'            : 'Cell 2',
    'DEVICE'         : 'Cell 1',
    'SESSION_START'  : 'Cell 1',
    'geo_features'   : 'Cell 3',
    'DTMPointNet2'   : 'Cell 4',
    'model'          : 'Cell 4',
    'GroundDataset'  : 'Cell 5',
}
_miss = {n: c for n, c in _NEED.items() if n not in dir()}
if _miss:
    _msg = '\n\nMissing names (kernel restart / skipped cell):\n'
    for n, c in _miss.items():
        _msg += f'  {n:<20s} <- defined in {c}\n'
    _msg += '\nACTION: Run -> Run All Cells  (Ctrl+Shift+F10)'
    raise NameError(_msg)

print('All dependencies present')

# ── AMP compatibility shim ────────────────────────────────────────────────────
# torch.amp.* = new API (>=2.3)   torch.cuda.amp.* = old API (<=2.2)
# This shim works on every version Kaggle has shipped.

def _scaler(enabled: bool):
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def _autocast(enabled: bool):
    try:
        return torch.amp.autocast('cuda', enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)


# ── Focal Loss ────────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, alpha: float, gamma: float, smooth: float):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        nc = logits.size(-1)
        if self.smooth > 0:
            soft = torch.zeros_like(logits).scatter_(
                1, targets.unsqueeze(1), 1.0)
            soft = soft * (1.0 - self.smooth) + self.smooth / nc
            ce   = -(soft * torch.log_softmax(logits, -1)).sum(-1)
        else:
            ce = nn.functional.cross_entropy(
                logits, targets, reduction='none')
        pt = torch.exp(-ce)
        at = torch.where(
            targets == 1,
            targets.new_full(targets.shape, self.alpha,       dtype=torch.float32),
            targets.new_full(targets.shape, 1.0 - self.alpha, dtype=torch.float32),
        )
        return (at * (1.0 - pt) ** self.gamma * ce).mean()


# ── Metrics ───────────────────────────────────────────────────────────────────

def _metrics(preds: torch.Tensor, labels: torch.Tensor):
    acc = (preds == labels).float().mean().item()
    tp  = ((preds==1) & (labels==1)).sum().item()
    fp  = ((preds==1) & (labels==0)).sum().item()
    fn  = ((preds==0) & (labels==1)).sum().item()
    p   = tp / (tp + fp + 1e-9)
    r   = tp / (tp + fn + 1e-9)
    f1  = 2*p*r / (p + r + 1e-9)
    return acc, p, r, f1


# ── DataLoaders ───────────────────────────────────────────────────────────────

def _loaders(cfg: dict):
    nw = cfg['num_workers']
    kw = dict(num_workers=nw, pin_memory=True,
              persistent_workers=(nw > 0),
              prefetch_factor=cfg['prefetch_factor'] if nw > 0 else None)
    tr_ds = GroundDataset(cfg, 'train', augment=True,
                          max_tiles=cfg['max_train_tiles'])
    va_ds = GroundDataset(cfg, 'val',   augment=False,
                          max_tiles=cfg['max_val_tiles'])
    tr_ld = DataLoader(tr_ds, batch_size=cfg['batch_size'],
                       shuffle=True,  drop_last=True,  **kw)
    va_ld = DataLoader(va_ds, batch_size=cfg['batch_size'],
                       shuffle=False, drop_last=False, **kw)
    print(f'  train: {len(tr_ds)} tiles -> {len(tr_ld)} batches')
    print(f'  val  : {len(va_ds)} tiles -> {len(va_ld)} batches')
    return tr_ld, va_ld


# ── Train ─────────────────────────────────────────────────────────────────────

def train(cfg: dict, model: nn.Module,
          device: torch.device):

    crit   = FocalLoss(cfg['focal_alpha'], cfg['focal_gamma'], cfg['label_smooth'])
    opt    = torch.optim.AdamW(model.parameters(),
                               lr=cfg['max_lr']/25.0,
                               weight_decay=cfg['weight_decay'])
    tr_ld, va_ld = _loaders(cfg)

    spe         = math.ceil(len(tr_ld) / cfg['grad_accum'])
    total_steps = cfg['epochs'] * spe

    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg['max_lr'], total_steps=total_steps,
        pct_start=0.25, anneal_strategy='cos',
        div_factor=25.0, final_div_factor=1e4)

    swa_m = AveragedModel(model)
    swa_s = SWALR(opt, swa_lr=cfg['swa_lr'])
    scaler = _scaler(cfg['use_amp'])

    history  = []
    best_acc = 0.0
    pat_cnt  = 0
    start_ep = 1
    t0       = time.time()

    # Auto-resume
    ckpt = Path(cfg['ckpt_path'])
    if cfg['resume'] and ckpt.exists():
        try:
            c = torch.load(str(ckpt), map_location=device)
            model.load_state_dict(c['model'])
            opt.load_state_dict(c['opt'])
            sched.load_state_dict(c['sched'])
            scaler.load_state_dict(c['scaler'])
            history  = c.get('history', [])
            best_acc = c.get('best_acc', 0.0)
            pat_cnt  = c.get('pat_cnt', 0)
            start_ep = c['epoch'] + 1
            print(f'Resumed epoch {c["epoch"]}  best={best_acc:.4f}')
        except Exception as e:
            print(f'Resume failed ({e}) -- starting fresh')

    print(f'\nTraining  epochs {start_ep}->{cfg["epochs"]}  '
          f'eff_batch={cfg["batch_size"]*cfg["grad_accum"]}\n')

    for ep in range(start_ep, cfg['epochs'] + 1):

        # Budget guard
        if (time.time() - SESSION_START) / 3600 >= SESSION_BUDGET_H:
            print(f'Budget reached at epoch {ep}. Checkpoint saved.')
            break

        # ── Train epoch ───────────────────────────────────────────────────────
        model.train()
        tr_loss = 0.0
        tr_p, tr_l = [], []
        opt.zero_grad()

        for step, (pts, lbs) in enumerate(
                tqdm(tr_ld, desc=f'Ep{ep:3d} train', leave=False, ncols=88)):
            pts = pts.to(device, non_blocking=True)
            lbs = lbs.to(device, non_blocking=True)
            with _autocast(cfg['use_amp']):
                loss = crit(model(pts).view(-1,2),
                            lbs.view(-1)) / cfg['grad_accum']
            scaler.scale(loss).backward()
            if (step+1) % cfg['grad_accum'] == 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
                scaler.step(opt); scaler.update(); opt.zero_grad()
                if ep <= cfg['swa_start']: sched.step()
            tr_loss += loss.item() * cfg['grad_accum']
            with torch.no_grad():
                tr_p.append(model(pts).argmax(-1).view(-1).cpu())
                tr_l.append(lbs.view(-1).cpu())

        ta, _, _, tf1 = _metrics(torch.cat(tr_p), torch.cat(tr_l))
        tr_loss /= len(tr_ld)

        # ── Validation ────────────────────────────────────────────────────────
        do_val = (ep % cfg['val_every'] == 0) or (ep == cfg['epochs'])
        va, vf1, vl = float('nan'), float('nan'), float('nan')

        if do_val:
            model.eval()
            vp, vla, vl_acc = [], [], 0.0
            with torch.no_grad():
                for pts, lbs in tqdm(va_ld, desc=f'Ep{ep:3d} val  ',
                                     leave=False, ncols=88):
                    pts = pts.to(device, non_blocking=True)
                    lbs = lbs.to(device, non_blocking=True)
                    with _autocast(cfg['use_amp']):
                        out = model(pts)
                        vl_acc += crit(out.view(-1,2), lbs.view(-1)).item()
                    vp.append(out.argmax(-1).view(-1).cpu())
                    vla.append(lbs.view(-1).cpu())
            va, _, _, vf1 = _metrics(torch.cat(vp), torch.cat(vla))
            vl = vl_acc / len(va_ld)
            if va > best_acc:
                best_acc = va; pat_cnt = 0
                torch.save(model.state_dict(), cfg['model_save_path'])
            else:
                pat_cnt += 1

        if ep >= cfg['swa_start']:
            swa_m.update_parameters(model); swa_s.step()

        star = '  BEST' if (do_val and va == best_acc) else ''
        print(f'Ep{ep:3d} tl={tr_loss:.4f} ta={ta:.4f} tf1={tf1:.4f} | '
              f'va={va:.4f} vf1={vf1:.4f} best={best_acc:.4f}{star}')

        history.append(dict(ep=ep,tl=tr_loss,ta=ta,tf1=tf1,
                            va=va,vl=vl,vf1=vf1))
        torch.save(dict(epoch=ep,model=model.state_dict(),
                        opt=opt.state_dict(),sched=sched.state_dict(),
                        scaler=scaler.state_dict(),history=history,
                        best_acc=best_acc,pat_cnt=pat_cnt),
                   str(cfg['ckpt_path']))

        if pat_cnt >= cfg['patience']:
            print(f'Early stop ep {ep}')
            break

    # SWA BN update
    print('Updating SWA BN...')
    bn_ld = DataLoader(GroundDataset(cfg,'train',augment=False),
                       batch_size=cfg['batch_size'],
                       num_workers=cfg['num_workers'], pin_memory=True)
    torch.optim.swa_utils.update_bn(bn_ld, swa_m, device=device)
    torch.save(swa_m.module.state_dict(), cfg['swa_save_path'])
    print(f'SWA saved: {cfg["swa_save_path"]}')

    with open(cfg['history_path'], 'w') as f:
        json.dump(history, f, indent=2)

    return history, best_acc, t0   # t0 returned so caller can compute elapsed


# ── Plot ──────────────────────────────────────────────────────────────────────

def _plot(history, path):
    vh  = [h for h in history if not math.isnan(h.get('va', math.nan))]
    aep = [h['ep'] for h in history]
    vep = [h['ep'] for h in vh]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(aep, [h['tl'] for h in history], label='train')
    if vh: ax[0].plot(vep, [h['vl'] for h in vh], label='val')
    ax[0].set(title='Focal Loss', xlabel='Epoch'); ax[0].legend()
    ax[1].plot(aep, [h['ta'] for h in history], label='train')
    if vh: ax[1].plot(vep, [h['va'] for h in vh], label='val')
    ax[1].axhline(0.95, ls='--', c='red', alpha=0.5, label='95% target')
    ax[1].set(title='Accuracy', xlabel='Epoch', ylim=(0.75,1.0)); ax[1].legend()
    plt.tight_layout()
    plt.savefig(path, dpi=120); plt.close()
    print(f'Curves: {path}')


# ── RUN ───────────────────────────────────────────────────────────────────────
history, best_acc, t0 = train(CFG, model, DEVICE)
elapsed_min = (time.time() - t0) / 60

print(f'\nDone  {elapsed_min:.1f} min')
print(f'Best val accuracy : {best_acc*100:.2f}%')
if best_acc >= 0.95:
    print('TARGET >95% : HIT')
else:
    gap = (0.95 - best_acc) * 100
    print(f'TARGET >95% : {gap:.1f}pp away')
    print("  In Cell 2: CFG['epochs']=120  or  CFG['focal_alpha']=0.80")

_plot(history, CFG['curves_path'])


In [ ]:
# Cell 7 — Threshold Optimisation
# Sweeps P(ground) thresholds on the val set for both models.
# Default threshold 0.50 is rarely optimal for imbalanced data.
# F1-optimal threshold typically adds +0.5-1.5% accuracy for free.

import json, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import DataLoader


def _sweep(model_path: str, cfg: dict, device: torch.device):
    if not Path(model_path).exists():
        return None
    m = DTMPointNet2(7).to(device)
    m.load_state_dict(torch.load(model_path, map_location=device))
    m.eval()

    va_ds = GroundDataset(cfg, 'val', augment=False)
    va_ld = DataLoader(va_ds, batch_size=cfg['batch_size'],
                       num_workers=cfg['num_workers'],
                       pin_memory=True, drop_last=False)

    all_p, all_l = [], []
    with torch.no_grad():
        for pts, lbs in tqdm(va_ld,
                             desc=f'Sweep {Path(model_path).stem}',
                             leave=False):
            out = m(pts.to(device))
            all_p.append(F.softmax(out, -1)[:, :, 1].view(-1).cpu().numpy())
            all_l.append(lbs.view(-1).numpy())

    probs = np.concatenate(all_p)
    labs  = np.concatenate(all_l)

    best_thr, best_f1, best_acc = 0.5, 0.0, 0.0
    curve = []
    for thr in np.arange(0.10, 0.90, 0.01):
        p   = (probs >= thr).astype(np.int64)
        tp  = ((p==1)&(labs==1)).sum()
        fp  = ((p==1)&(labs==0)).sum()
        fn  = ((p==0)&(labs==1)).sum()
        pr  = tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9)
        f1  = 2*pr*rc/(pr+rc+1e-9)
        acc = (p==labs).mean()
        curve.append((float(thr), float(f1), float(acc)))
        if f1 > best_f1:
            best_f1=f1; best_thr=float(thr); best_acc=float(acc)

    return dict(model_path=model_path, threshold=best_thr,
                val_accuracy=best_acc, val_f1=best_f1, curve=curve)


results = {}
for label, path in [('best_epoch', CFG['model_save_path']),
                     ('swa',        CFG['swa_save_path'])]:
    r = _sweep(path, CFG, DEVICE)
    if r:
        results[label] = r
        print(f'[{label}]  thr={r["threshold"]:.2f}  '
              f'acc={r["val_accuracy"]*100:.2f}%  f1={r["val_f1"]:.4f}')

winner = max(results, key=lambda k: results[k]['val_accuracy'])
best   = results[winner]
print(f'\nWinner: {winner}  acc={best["val_accuracy"]*100:.2f}%')

# Plot
c   = best['curve']
fig, ax = plt.subplots(figsize=(8,4))
ax.plot([x[0] for x in c],[x[1] for x in c],label='F1')
ax.plot([x[0] for x in c],[x[2] for x in c],label='Acc')
ax.axvline(best['threshold'],ls='--',c='red',
           label=f'thr={best["threshold"]:.2f}')
ax.axhline(0.95,ls=':',c='orange',alpha=0.7,label='95% target')
ax.set(xlabel='Threshold',ylim=(0.8,1.0),title='Val threshold sweep')
ax.legend(); plt.tight_layout()
plt.savefig(str(Path(CFG['logs_dir'])/'threshold_curve.png'),dpi=120)
plt.close()

with open(CFG['threshold_path'],'w') as f:
    json.dump(dict(model=winner, model_path=best['model_path'],
                   threshold=best['threshold'],
                   val_accuracy=best['val_accuracy'],
                   val_f1=best['val_f1']), f, indent=2)

print(f'Saved: {CFG["threshold_path"]}')
print('Cell 7 done -- proceed to Cell 8')


In [ ]:
# Cell 8 — Package all outputs for download
import zipfile, json
from pathlib import Path

with open(CFG['threshold_path']) as f:
    meta = json.load(f)

files = [
    meta['model_path'],
    CFG['model_save_path'],
    CFG['swa_save_path'],
    CFG['threshold_path'],
    CFG['history_path'],
    CFG['curves_path'],
    str(Path(CFG['logs_dir']) / 'threshold_curve.png'),
]

out_zip = Path('/kaggle/working/dtm_outputs.zip')
with zipfile.ZipFile(str(out_zip), 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        if Path(p).exists():
            zf.write(p, Path(p).name)
            print(f'  packed {Path(p).name}  '
                  f'({Path(p).stat().st_size/1e3:.0f} KB)')

print(f'\ndtm_outputs.zip  ({out_zip.stat().st_size/1e6:.1f} MB)')
print()
print('NEXT STEPS:')
print('  Kaggle sidebar -> Output -> dtm_outputs.zip -> Download')
print('  Extract -> copy to Terra Pravah backend/models/')
print('    best_model.pth         -> backend/models/')
print('    optimal_threshold.json -> backend/models/')
